# Phase 2P - Is it leakage, or just typical images?

**Run on:** Kaggle or Colab, free T4. ~40 min for 25 training runs.

---

### The objection this answers

The deduplication result removes training images that are near-duplicates of
test images, and compares against removing the same *number* at random. But
those images are not a random sample: they are the central members of dense
clusters. **Removing cluster-central images costs accuracy in any dataset,
leaked or not.** So the size-matched control does not separate

  (a) the cost of losing images that leak information about the test set, from
  (b) the cost of losing typical, on-manifold training examples.

### The design

Hold typicality fixed; vary only whether the held-out set is evaluated.

| arm | held-out set | removed from training | evaluated on |
|---|---|---|---|
| baseline | test fold | nothing | test fold |
| **test-dedup** | test fold | near-duplicates of the test fold | test fold |
| test-random | test fold | same count, random | test fold |
| **ref-dedup** | R, a random training subset of the same size | R, plus near-duplicates of R | test fold |
| ref-random | R | R, plus same count random | test fold |

R plays exactly the role the test fold plays: same size, held out of training,
near-duplicates deleted. The difference is that R is never evaluated. Both
dedup arms therefore delete cluster-central images at comparable rates; only
one of them deletes images that carry information about what is scored.

### What the outcome means, fixed before the run

- **effect_test substantially exceeds effect_reference:** the cost is specific
  to duplicates of the evaluated set. That is leakage, and the causal wording
  is earned.
- **effect_test is comparable to effect_reference:** the cost is generic to
  removing typical images. The causal claim fails and the paper must retreat to
  "consistent with near-duplicate leakage" throughout.

We report whichever occurs.

## 1. Environment, data, similarity

In [ ]:
import subprocess, sys
for pkg in ["opencv-python-headless", "tabulate", "kagglehub"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg],
                   check=False)

import os, json, time, shutil, random, gc
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             balanced_accuracy_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("GPUs:", tf.config.list_physical_devices("GPU"))

IN_KAGGLE = os.path.exists("/kaggle/working")
WORK    = "/kaggle/working" if IN_KAGGLE else "/content"
SCRATCH = "/kaggle/temp"    if IN_KAGGLE else "/content"
try:
    os.makedirs(SCRATCH, exist_ok=True)
except OSError:
    SCRATCH = "/tmp"; os.makedirs(SCRATCH, exist_ok=True)

RESULTS_DIR = f"{WORK}/fyp_phase2p_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
N_FOLDS, THRESHOLD = 5, 0.98
RESULTS = {"seed": SEED, "n_folds": N_FOLDS, "threshold": THRESHOLD}

def save_json():
    with open(f"{RESULTS_DIR}/results.json", "w") as f:
        json.dump(RESULTS, f, indent=2, default=float)
print("outputs ->", RESULTS_DIR)

In [ ]:
FOLDERS = {"Benign": "Bengin cases", "Malignant": "Malignant cases",
           "Normal": "Normal cases"}
DATA_ROOT = None
if os.path.isdir("/kaggle/input"):
    hits = [d for d, _, _ in os.walk("/kaggle/input")
            if os.path.basename(d) == "Malignant cases"]
    if hits:
        DATA_ROOT = os.path.dirname(hits[0])
if DATA_ROOT is None:
    import kagglehub
    DL = kagglehub.dataset_download("hamdallak/the-iqothnccd-lung-cancer-dataset")
    cands = [d for d, _, _ in os.walk(DL) if os.path.basename(d) == "Malignant cases"]
    DATA_ROOT = os.path.dirname(cands[0])
print("data:", DATA_ROOT)

SPLIT_URL = ("https://raw.githubusercontent.com/haseebkhan9081/"
             "iqothnccd-leakage-audit/main/split_seed42.csv")
subprocess.run(["wget", "-q", "-O", f"{SCRATCH}/split_seed42.csv", SPLIT_URL],
               check=True)
df = pd.read_csv(f"{SCRATCH}/split_seed42.csv")
df["path"] = [os.path.join(DATA_ROOT, FOLDERS[l], f)
              for l, f in zip(df["label"], df["file"])]
assert all(os.path.exists(p) for p in df["path"])
y_all = df["y"].to_numpy()

thumbs = []
for p in tqdm(df["path"], desc="thumbnails"):
    g = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, (64, 64), interpolation=cv2.INTER_AREA).astype(np.float32).ravel()
    g -= g.mean()
    n = np.linalg.norm(g)
    thumbs.append(g / n if n > 0 else g)
T = np.stack(thumbs)
S = T @ T.T
np.fill_diagonal(S, 0.0)
print("images:", len(df))
save_json()

## 2. The five conditions

`R` is drawn from the training partition only, and is the same size as that
fold's test set. It is removed from training in both reference arms, exactly as
the test fold is absent from training in the test arms.

In [ ]:
folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                             random_state=SEED).split(df, y_all))
rng = np.random.default_rng(SEED)

CONDITIONS = {"baseline": [], "test-dedup": [], "test-random": [],
              "ref-dedup": [], "ref-random": []}
stats_rows = []

for k, (tr, te) in enumerate(folds):
    CONDITIONS["baseline"].append((tr, te))

    # --- test arm: remove training images near-duplicate to the TEST fold
    hit_te = (S[np.ix_(tr, te)] >= THRESHOLD).any(axis=1)
    n_te = int(hit_te.sum())
    CONDITIONS["test-dedup"].append((tr[~hit_te], te))
    CONDITIONS["test-random"].append(
        (np.sort(rng.choice(tr, size=len(tr) - n_te, replace=False)), te))

    # --- reference arm: R is a held-out TRAINING subset of test-fold size
    R = rng.choice(tr, size=len(te), replace=False)
    rest = np.setdiff1d(tr, R)
    hit_R = (S[np.ix_(rest, R)] >= THRESHOLD).any(axis=1)
    n_R = int(hit_R.sum())
    CONDITIONS["ref-dedup"].append((rest[~hit_R], te))
    CONDITIONS["ref-random"].append(
        (np.sort(rng.choice(rest, size=len(rest) - n_R, replace=False)), te))

    stats_rows.append({"fold": k, "n_train": int(len(tr)), "n_test": int(len(te)),
                       "removed_by_test": n_te,
                       "pct_removed_by_test": round(100 * n_te / len(tr), 2),
                       "reference_size": int(len(R)),
                       "removed_by_reference": n_R,
                       "pct_removed_by_reference": round(100 * n_R / len(rest), 2)})
    print(f"fold {k}: test-neighbours {n_te:4d} ({100*n_te/len(tr):.1f}%) | "
          f"reference-neighbours {n_R:4d} ({100*n_R/len(rest):.1f}%)")

RESULTS["removal_stats"] = stats_rows
for name, cf in CONDITIONS.items():
    for k in range(N_FOLDS):
        assert np.array_equal(cf[k][1], folds[k][1]), f"{name} test fold differs"
print("\ntest folds identical across all five conditions: OK")
print("If the two removal percentages are similar, the two dedup arms delete")
print("cluster-central images at comparable rates, which is what makes the")
print("comparison fair.")
save_json()

## 3. Run

In [ ]:
SIZE, EPOCHS, BATCH, LR = 224, 30, 16, 1e-4
X = np.empty((len(df), SIZE, SIZE, 3), np.float32)
for i, p in enumerate(tqdm(df["path"], desc=f"load {SIZE}px")):
    X[i] = np.asarray(Image.open(p).convert("RGB").resize((SIZE, SIZE),
                                                          Image.BILINEAR), np.float32)

def build():
    inp = layers.Input((SIZE, SIZE, 3))
    x = keras.Sequential([layers.RandomFlip("horizontal"),
                          layers.RandomRotation(0.05),
                          layers.RandomZoom(0.15, 0.15)], name="aug")(inp)
    base = keras.applications.ResNet50(include_top=False, weights="imagenet",
                                       input_shape=(SIZE, SIZE, 3))
    base.trainable = True
    for layer in base.layers[:-20]:
        layer.trainable = False
    x = base(keras.applications.resnet50.preprocess_input(x), training=False)
    x = layers.Dropout(0.3)(layers.GlobalAveragePooling2D()(x))
    return keras.Model(inp, layers.Dense(3, activation="softmax")(x))

def evaluate(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0)
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1.mean())}

rows, t0 = [], time.time()
for cond, cf in CONDITIONS.items():
    for k, (tr, te) in enumerate(cf):
        keras.backend.clear_session()
        tf.random.set_seed(SEED + k)
        counts = np.bincount(y_all[tr], minlength=3)
        cw = {i: float(len(tr) / (3 * c)) if c else 0.0 for i, c in enumerate(counts)}
        model = build()
        model.compile(optimizer=keras.optimizers.Adam(LR),
                      loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        model.fit(X[tr], y_all[tr], epochs=EPOCHS, batch_size=BATCH,
                  class_weight=cw, verbose=0)
        m = evaluate(y_all[te], model.predict(X[te], verbose=0).argmax(1))
        m.update({"condition": cond, "fold": k, "n_train": int(len(tr))})
        rows.append(m)
        print(f"{cond:14s} fold {k} | bal {m['balanced_accuracy']:.3f} "
              f"(n_train {len(tr)})")
        del model; gc.collect()

cv = pd.DataFrame(rows)
cv.to_csv(f"{RESULTS_DIR}/cv_per_fold.csv", index=False)
RESULTS["per_fold"] = cv.to_dict("records")
print(f"\ntotal {(time.time()-t0)/60:.1f} min over {len(cv)} runs")
save_json()

## 4. The answer

In [ ]:
from scipy import stats

def bal(cond):
    sel = cv[cv.condition == cond].sort_values("fold")
    return sel["balanced_accuracy"].to_numpy()

def interval(d, rho):
    k = len(d)
    se = np.sqrt((1.0 / k + rho) * d.var(ddof=1))
    crit = stats.t.ppf(0.975, k - 1)
    p = float(2 * stats.t.sf(abs(d.mean() / se), k - 1)) if se else np.nan
    return d.mean(), d.mean() - crit * se, d.mean() + crit * se, p

rho = float(np.mean([r["n_test"] / r["n_train"] for r in stats_rows]))
eff_test = bal("test-random") - bal("test-dedup")
eff_ref  = bal("ref-random")  - bal("ref-dedup")

mt, lt, ht, pt = interval(eff_test, rho)
mr, lr, hr, pr = interval(eff_ref, rho)
diff = eff_test - eff_ref
md_, ld, hd, pd_ = interval(diff, rho)

print(f"baseline balanced accuracy      : {bal('baseline').mean():.4f}\n")
print(f"TEST arm      dedup {bal('test-dedup').mean():.4f}  "
      f"random {bal('test-random').mean():.4f}")
print(f"  size-controlled effect        : {mt:+.4f}  ({lt:+.3f}, {ht:+.3f})  p={pt:.4f}")
print(f"REFERENCE arm dedup {bal('ref-dedup').mean():.4f}  "
      f"random {bal('ref-random').mean():.4f}")
print(f"  size-controlled effect        : {mr:+.4f}  ({lr:+.3f}, {hr:+.3f})  p={pr:.4f}")
print()
print(f"LEAKAGE-SPECIFIC EXCESS (test - reference): {md_:+.4f}  "
      f"({ld:+.3f}, {hd:+.3f})  p={pd_:.4f}")
print()
if ld > 0:
    print("The test arm costs significantly MORE than the reference arm.")
    print("Removing near-duplicates of the EVALUATED set is worse than removing")
    print("equally typical near-duplicates of an unevaluated set. That is leakage,")
    print("and the causal wording is earned.")
else:
    print("The two arms are not distinguishable. The cost is generic to removing")
    print("typical training images, NOT specific to the evaluated set. The causal")
    print("claim does not hold; the paper must say 'consistent with' throughout.")

RESULTS["verdict"] = {
    "baseline": float(bal("baseline").mean()), "rho": rho,
    "effect_test": float(mt), "effect_test_ci": [float(lt), float(ht)],
    "effect_test_p": float(pt),
    "effect_reference": float(mr), "effect_reference_ci": [float(lr), float(hr)],
    "effect_reference_p": float(pr),
    "leakage_excess": float(md_), "leakage_excess_ci": [float(ld), float(hd)],
    "leakage_excess_p": float(pd_),
    "leakage_specific": bool(ld > 0)}
save_json()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
order = ["baseline", "test-dedup", "test-random", "ref-dedup", "ref-random"]
cols = ["#555555", "#c0392b", "#e59866", "#2471a3", "#85c1e9"]
for i, (c, col) in enumerate(zip(order, cols)):
    v = cv[cv.condition == c]["balanced_accuracy"]
    ax.scatter([i] * len(v), v, color=col, alpha=0.35, s=24, linewidths=0)
    ax.errorbar(i, v.mean(), yerr=v.std(), fmt="o", color=col, markersize=9,
                capsize=5, linewidth=1.8)
ax.set_xticks(range(5)); ax.set_xticklabels(order, fontsize=9, rotation=12)
ax.set_ylabel("balanced accuracy"); ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
ax.set_title("Same test folds. Reference arm removes equally typical images\n"
             "that are near-duplicates of an UNevaluated held-out set.",
             fontsize=10)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig_typicality_control.png", dpi=200)
plt.show()

path = shutil.make_archive(f"{WORK}/fyp_phase2p_results", "zip", RESULTS_DIR)
print("archive:", path)